# Train LSTM Incident Classifier

This notebook trains an LSTM-based department classifier using `data/processed/incidents_train.csv`.
It builds a vocabulary from the incident text, trains a PyTorch LSTM model, and reports classification metrics with a confusion matrix.


In [6]:
from collections import Counter
from pathlib import Path
import random
import re

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "processed" / "incidents_train.csv").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed/incidents_train.csv"
    )

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_DATASET_PATH = PROCESSED_DATA_DIR / "incidents_train.csv"
VALIDATION_DATASET_PATH = PROCESSED_DATA_DIR / "incidents_validation.csv"
TEST_DATASET_PATH = PROCESSED_DATA_DIR / "incidents_test.csv"
LABEL_MAPPING_PATH = PROCESSED_DATA_DIR / "department_label_mapping.csv"

TEXT_COLUMN = "feature_concatanation"
LABEL_COLUMN = "department_label"
LABEL_NAME_COLUMN = "assigned_department"
RANDOM_SEED = 42
VOCAB_SIZE = 12000
MAX_LENGTH = 64
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.2
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-3
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [7]:
train_df = pd.read_csv(TRAIN_DATASET_PATH)
validation_df = pd.read_csv(VALIDATION_DATASET_PATH)
test_df = pd.read_csv(TEST_DATASET_PATH)
label_mapping_df = pd.read_csv(LABEL_MAPPING_PATH).sort_values(LABEL_COLUMN)

required_columns = {TEXT_COLUMN, LABEL_COLUMN, LABEL_NAME_COLUMN}
for dataset_name, dataset_df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():
    missing_columns = required_columns - set(dataset_df.columns)
    if missing_columns:
        raise ValueError(
            f"{dataset_name} dataset is missing required columns: {sorted(missing_columns)}"
        )


def prepare_dataset(dataset_df):
    prepared_df = dataset_df[[TEXT_COLUMN, LABEL_COLUMN, LABEL_NAME_COLUMN]].dropna().copy()
    prepared_df[TEXT_COLUMN] = prepared_df[TEXT_COLUMN].astype(str).str.strip()
    prepared_df = prepared_df[prepared_df[TEXT_COLUMN] != ""]
    prepared_df[LABEL_COLUMN] = prepared_df[LABEL_COLUMN].astype(int)
    return prepared_df


train_df = prepare_dataset(train_df)
validation_df = prepare_dataset(validation_df)
test_df = prepare_dataset(test_df)

label_mapping = dict(
    zip(label_mapping_df[LABEL_COLUMN].astype(int), label_mapping_df[LABEL_NAME_COLUMN])
)

print(f"Loaded {len(train_df)} train rows from {TRAIN_DATASET_PATH}")
print(f"Loaded {len(validation_df)} validation rows from {VALIDATION_DATASET_PATH}")
print(f"Loaded {len(test_df)} test rows from {TEST_DATASET_PATH}")
print(f"Number of departments: {len(label_mapping)}")
train_df.head()


Loaded 2520 train rows from /home/lakshan/ssp/IMS/data/processed/incidents_train.csv
Loaded 540 validation rows from /home/lakshan/ssp/IMS/data/processed/incidents_validation.csv
Loaded 540 test rows from /home/lakshan/ssp/IMS/data/processed/incidents_test.csv
Number of departments: 12


,feature_concatanation,department_label,assigned_department
0,Ward Sister - Medical Please send porter team ...,11,Transport
1,Pharmacy Manager Antibiotic bulk store damp; h...,7,Quality Assurance department
2,"Labour Room Midwife Between cases, blood splas...",4,Housekeeping Department
3,Pharmacy Intern - Inpatient Glucometer strips ...,10,Supply Department
4,"Head, Physiotherapy TENS units due for annual ...",0,Biomedical Engineering


In [8]:
X_train = train_df[TEXT_COLUMN]
y_train = train_df[LABEL_COLUMN]
X_validation = validation_df[TEXT_COLUMN]
y_validation = validation_df[LABEL_COLUMN]
X_test = test_df[TEXT_COLUMN]
y_test = test_df[LABEL_COLUMN]


def tokenize(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", text.lower())


token_counter = Counter()
for text in X_train:
    token_counter.update(tokenize(text))

most_common_tokens = token_counter.most_common(VOCAB_SIZE - 2)
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token, _ in most_common_tokens:
    vocab[token] = len(vocab)


def encode_text(text: str, vocab: dict[str, int], max_length: int) -> list[int]:
    token_ids = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokenize(text)]
    if not token_ids:
        token_ids = [vocab[UNK_TOKEN]]
    return token_ids[:max_length]


class IncidentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded_text = encode_text(self.texts.iloc[idx], self.vocab, self.max_length)
        return {
            "input_ids": torch.tensor(encoded_text, dtype=torch.long),
            "label": torch.tensor(int(self.labels.iloc[idx]), dtype=torch.long),
        }


def collate_batch(batch):
    sequences = [item["input_ids"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=vocab[PAD_TOKEN])
    return {
        "input_ids": padded_sequences,
        "lengths": lengths,
        "labels": labels,
    }


train_dataset = IncidentDataset(X_train, y_train, vocab, MAX_LENGTH)
validation_dataset = IncidentDataset(X_validation, y_validation, vocab, MAX_LENGTH)
test_dataset = IncidentDataset(X_test, y_test, vocab, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train), len(X_validation), len(X_test)],
        "source": [
            str(TRAIN_DATASET_PATH),
            str(VALIDATION_DATASET_PATH),
            str(TEST_DATASET_PATH),
        ],
    }
)


,split,rows,source
0,train,2520,/home/lakshan/ssp/IMS/data/processed/incidents...
1,validation,540,/home/lakshan/ssp/IMS/data/processed/incidents...
2,test,540,/home/lakshan/ssp/IMS/data/processed/incidents...


In [9]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, pad_index, num_layers=1, dropout=0.0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_index)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden_state, _) = self.lstm(packed)
        final_hidden = hidden_state[-1]
        logits = self.classifier(self.dropout(final_hidden))
        return logits


num_labels = len(label_mapping)
model = LSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_labels=num_labels,
    pad_index=vocab[PAD_TOKEN],
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def run_epoch(model, data_loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    predictions = []
    labels = []

    context_manager = torch.enable_grad() if is_training else torch.no_grad()
    with context_manager:
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            lengths = batch["lengths"].to(device)
            batch_labels = batch["labels"].to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, batch_labels)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            predictions.extend(torch.argmax(logits, dim=1).cpu().tolist())
            labels.extend(batch_labels.cpu().tolist())

    average_loss = total_loss / max(len(data_loader), 1)
    accuracy = accuracy_score(labels, predictions) if labels else 0.0
    return average_loss, accuracy, predictions, labels


history = []
for epoch in range(EPOCHS):
    train_loss, train_accuracy, _, _ = run_epoch(model, train_loader, criterion, optimizer)
    validation_loss, validation_accuracy, _, _ = run_epoch(model, validation_loader, criterion)

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
    }
    history.append(epoch_result)
    print(epoch_result)

history_df = pd.DataFrame(history)
history_df


{'epoch': 1, 'train_loss': 2.4097637254980544, 'train_accuracy': 0.19960317460317462, 'validation_loss': 2.2232869092155907, 'validation_accuracy': 0.36666666666666664}
{'epoch': 2, 'train_loss': 1.6434318415726288, 'train_accuracy': 0.5222222222222223, 'validation_loss': 1.263229044044719, 'validation_accuracy': 0.6333333333333333}
{'epoch': 3, 'train_loss': 0.8971803029881248, 'train_accuracy': 0.7432539682539683, 'validation_loss': 0.9427483081817627, 'validation_accuracy': 0.7074074074074074}
{'epoch': 4, 'train_loss': 0.4678977890105187, 'train_accuracy': 0.8690476190476191, 'validation_loss': 0.8262124692692476, 'validation_accuracy': 0.7444444444444445}
{'epoch': 5, 'train_loss': 0.23707805451335787, 'train_accuracy': 0.9400793650793651, 'validation_loss': 0.7753942328340867, 'validation_accuracy': 0.774074074074074}
{'epoch': 6, 'train_loss': 0.12587828552232513, 'train_accuracy': 0.9714285714285714, 'validation_loss': 0.8079098340343026, 'validation_accuracy': 0.75925925925925

,epoch,train_loss,train_accuracy,validation_loss,validation_accuracy
0,1,2.409764,0.199603,2.223287,0.366667
1,2,1.643432,0.522222,1.263229,0.633333
2,3,0.897180,0.743254,0.942748,0.707407
3,4,0.467898,0.869048,0.826212,0.744444
4,5,0.237078,0.940079,0.775394,0.774074
5,6,0.125878,0.971429,0.807910,0.759259
6,7,0.054744,0.992063,0.816327,0.779630
7,8,0.022761,0.997619,0.845949,0.785185
8,9,0.010909,0.999206,0.924881,0.796296
9,10,0.006360,0.999603,0.928186,0.774074


In [10]:
test_loss, test_accuracy, test_predictions, test_labels = run_epoch(model, test_loader, criterion)

ordered_labels = sorted(label_mapping)
target_names = [label_mapping[label] for label in ordered_labels]

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nTest classification report:\n")
print(
    classification_report(
        test_labels,
        test_predictions,
        labels=ordered_labels,
        target_names=target_names,
        zero_division=0,
    )
)

confusion_matrix_df = pd.DataFrame(
    confusion_matrix(test_labels, test_predictions, labels=ordered_labels),
    index=[f"actual_{label_mapping[label]}" for label in ordered_labels],
    columns=[f"pred_{label_mapping[label]}" for label in ordered_labels],
)

print("\nConfusion matrix:\n")
confusion_matrix_df


Test loss: 0.9232
Test accuracy: 0.7519

Test classification report:

                              precision    recall  f1-score   support

      Biomedical Engineering       0.61      0.86      0.71        42
   Dietary and Food Services       0.49      0.93      0.65        45
         Facility Management       0.71      0.73      0.72        48
         Finance and Account       0.77      0.78      0.78        51
     Housekeeping Department       0.94      0.70      0.80        46
             Human Resources       0.82      0.73      0.78        45
                          IT       0.80      0.62      0.70        45
Quality Assurance department       0.84      0.81      0.83        32
                   Reception       0.88      0.84      0.86        45
                    Security       0.88      0.72      0.79        40
           Supply Department       0.82      0.67      0.74        55
                   Transport       0.88      0.65      0.75        46

                  

,pred_Biomedical Engineering,pred_Dietary and Food Services,pred_Facility Management,pred_Finance and Account,pred_Housekeeping Department,pred_Human Resources,pred_IT,pred_Quality Assurance department,pred_Reception,pred_Security,pred_Supply Department,pred_Transport
actual_Biomedical Engineering,36,2,3,0,0,0,0,0,0,0,0,1
actual_Dietary and Food Services,1,42,1,0,0,0,0,1,0,0,0,0
actual_Facility Management,3,6,35,0,1,0,0,1,0,0,2,0
actual_Finance and Account,1,3,0,40,0,3,2,0,0,0,1,1
actual_Housekeeping Department,2,5,3,0,32,0,2,0,0,1,1,0
actual_Human Resources,4,3,0,3,0,33,0,0,1,0,0,1
actual_IT,6,4,2,1,0,0,28,0,2,0,2,0
actual_Quality Assurance department,0,1,2,0,0,1,1,26,0,0,1,0
actual_Reception,0,1,1,2,0,0,2,0,38,1,0,0
actual_Security,0,4,2,0,1,1,0,2,1,29,0,0
